In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df = spark.read \
    .option("header", "true") \
    .csv('fhv_tripdata_2019-10.csv.gz')

In [ ]:
df.show(5)

## Question 2: Average size of the Parquet file

In [ ]:
df.repartition(6).write.parquet('../../data/pq/fhv/2019/10')

## Question 3: How many taxi trips were there on the 15th of October?

In [ ]:
df = spark.read.parquet('../../data/pq/fhv/2019/10')

In [ ]:
df.filter(df['pickup_datetime'].like('2019-10-15%')).count()

## Question 4: What is the length of the longest trip in the dataset in hours?

In [ ]:
df = spark.read.parquet('../../data/pq/fhv/2019/10')

In [ ]:
# Convert the pickup and dropoff datetime to datetime
df = df.withColumn('pickup_datetime', df['pickup_datetime'].cast('timestamp'))
df = df.withColumn('dropoff_datetime', df['dropoff_datetime'].cast('timestamp'))

In [ ]:
df = df.withColumn('trip_duration', df['dropoff_datetime'].cast('long') - df['pickup_datetime'].cast('long'))

In [ ]:
# Convert the trip duration to hours
max_trip_seconds = df.agg({'trip_duration': 'max'})
max_trip_hours = max_trip_seconds.collect()[0][0] / 3600
print(max_trip_hours)

In [ ]:
# Get the trip with the longest duration
df.filter(df['trip_duration'] == max_trip_seconds.collect()[0][0]).show()

## Question 5: Least frequent pickup location zone

Using the zone lookup data and the FHV October 2019 data, what is the name of the LEAST frequent pickup location Zone?

In [ ]:
zones_df = spark.read \
    .parquet('../../installation_test/zones')

In [ ]:
zones_df.show(5)

In [ ]:
df = spark.read.parquet('../../data/pq/fhv/2019/10')

In [ ]:
df.groupBy('PULocationID').count().orderBy('count').show(5)

In [ ]:
df.join(zones_df, df['PULocationID'] == zones_df['LocationID']).groupBy('Zone').count().orderBy('count').show(5)